# Edge–Fog–Cloud: среда и проверка конфигурации

**Текущий этап — имитация, не нейросетевой прогноз.** Требования к статистической карте: **p95 ≤ 200 мс И потери ≤ 0,1%**. Неполная карта не считается полной.

Начните с **Среда выполнения → Выполнить все**. Один начальный короткий опыт выполняется явно; затем расчёт — только по кнопке. Полная сетка и обучение автоматически не запускаются. Все числа синтетические.

p95 — 95-й перцентиль задержек завершённых сообщений. Потери считаются по всем поступившим. H(t) — доля всех поступивших, завершённых не позднее t после собственного поступления. Доверительный интервал (ДИ) оценивает неопределённость; один демонстрационный опыт его не даёт.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
while root != root.parent and not (root/'efc/model.py').exists():
    root = root.parent
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not (root/'efc/model.py').exists():
    if not IN_COLAB:
        raise RuntimeError('Запустите тетрадь из копии репозитория или откройте её в Colab.')
    root = Path('/content/fefc-r14')
    if not root.exists():
        subprocess.run(['git','clone','--branch','night/fefc-v3','--single-branch',
                        'https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(root)],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'requirements.txt')],check=True)
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
os.chdir(root)
sys.path.insert(0,str(root))
if (root/'.git').exists():
    SOURCE_SHA = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
else:
    SOURCE_SHA = os.environ.get('FEFC_SOURCE_SHA','Локальный архив: SHA смотрите в реестре источника')
from efc.dashboard import Dashboard, architecture_figure, manual_case, manual_figure, shutdown_case
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
print('Источник:',SOURCE_SHA,'; Python:',sys.version.split()[0])
print('Используется текущая локальная копия; обновление ветки не выполняется молча.')

## 1. Где обрабатывается сообщение

Слева направо — путь уже созданного сообщения. В Fog ограничены **места ожидания B**; ещё одно сообщение может обслуживаться. Заполнение буфера означает потерю сообщения, а не отказ датчика. Основная среда: 1000 датчиков и 40 облачных серверов; число Fog меняется. Нейросеть к этому пути не подключена.

In [ ]:
fig = architecture_figure()
display(fig)
plt.close(fig)

## 2. Три настройки и одна кнопка

**λ** — общий поток сообщений в секунду; **Fog** — число туманных узлов; **B** — вместимость ожидания каждого узла. ρ50 — опорная предложенная нагрузка при 50 Fog, не измеренная занятость.

Кнопка выполняет **600 сообщений, seed 42, без прогрева**: это иллюстрация очередей, не итоговая научная оценка. ДИ не выдумываются. Отдельная карточка читает сохранённую статистику только из подходящего checkpoint. Отсутствующие данные — «не рассчитано», не ноль.

Для собственного уже рассчитанного архива укажите каталог в переменной среды `FEFC_CHECKPOINT`; по умолчанию `results/r14/night-map`. Архив берётся из конкретного Actions, не из старых CSV. Перемещение настроек не запускает вычисления.

In [ ]:
checkpoint = Path(os.environ.get('FEFC_CHECKPOINT','results/r14/night-map'))
panel = Dashboard(checkpoint)
display(panel.widget)
# Один явно обозначенный начальный демонстрационный опыт.
panel.button.click()
assert panel.last_error is None, panel.last_error
assert panel.calls == 1
assert panel.last['result']['display_summary']['intervals'] is None
print('Короткий опыт выполнен. Полная статистическая сетка не запускалась.')

**Как читать вывод.** p95 относится к завершённым, потери и H(t) — ко всем созданным сообщениям. Предельная высота H(t) после дообслуживания равна 1 минус доля потерь. Шесть средних составляющих вычислены по одной совокупности завершённых. График очереди — только текущий короткий опыт.

Сохранённая карта и графики p95/потерь с ДИ показывают исключительно загруженные итоговые оценки. Пустые места не соединяются и не означают допустимость. Короткий опыт не записывает клетки карты и не подтверждает минимум.

## 3. Пять сообщений: ручной ответ и модель

Отдельный синтетический пример: один Edge/Fog/Cloud, B=2, поступления 0, 1, 2, 3, 10 мс. Сервисы: Edge=1, Edge–Fog=1, Fog=10, Fog–Cloud=1, Cloud=1 мс. Ручные ожидания Fog: **0, 9, 18, потеря, 20 мс**; сквозные задержки: **14, 23, 32, потеря, 34 мс**.

Пятое сообщение приходит к Fog ровно при освобождении; сначала завершается обслуживание. Четвёртое потеряно и не получает нулевую задержку. Отрезки показывают шесть неперекрывающихся составляющих.

In [ ]:
import pandas as pd
manual, hand_wait, hand_delay = manual_case()
display(pd.DataFrame([{'Сообщение':i+1,'Ожидание, ручное (мс)':hand_wait[i],
    'Ожидание, модель (мс)':t.get('fog_wait_ms'), 'Задержка, ручная (мс)':hand_delay[i],
    'Задержка, модель (мс)':t.get('latency_ms'), 'Статус':t['status']} for i,t in enumerate(manual['tasks'])]))
fig = manual_figure(manual)
display(fig)
plt.close(fig)
print('Ручные ответы и сумма шести составляющих проверены по целым наносекундным тикам.')

## 4. Ручное отключение — не прогноз и не потеря в Fog

Кнопка в раскрывающемся блоке панели запускает отдельный пример: два датчика, один Fog, отключение датчика 0 в момент 3 мс. На совпадении времён отключение предшествует созданию нового сообщения. Уже созданные сообщения, даже ещё ожидающие, сохраняются и дообслуживаются.

Исходные номера сохранены отдельным соответствием. Не созданные после команды сообщения не включаются в число поступивших или в потери Fog. Иллюстрация не меняет основную карту. Следующая ячейка проверяет обработчик кнопки, не диагностирует реальный отказ.

In [ ]:
panel.off_button.click()
assert panel.shutdown is not None
assert panel.shutdown['original_event_ids'] == [0,1,2,4]
assert panel.shutdown['not_created_ids'] == [3,5]
assert panel.shutdown['result']['metrics']['completed'] == 4
assert panel.shutdown['result']['metrics']['dropped'] == 0
print('Кнопка проверена: 4 созданных сообщения завершены, 2 не созданы; потери Fog=0.')

## 5. Что подтверждается и где

| Утверждение / граница вывода | Ячейка | Источник |
|---|---|---|
| Шесть стадий и ручные ответы пяти сообщений | `r14-main-manual` | `dashboard.manual_case`, TASK §5; синтетический оракул |
| Короткий опыт не является статистической допустимостью | `r14-main-panel` | `display_summary.statistical_status`; ДИ отсутствуют |
| p95, потери, H(200) и ДИ сохранённой клетки | `r14-main-panel` | Только `checkpoint/cells/<key>/main.json`, хеш из map; при отсутствии — не рассчитано |
| Условия p95 и потерь проверяются совместно | `r14-main-panel` | Неизменённый `statistics.classify`, TASK §4 |
| Отключение сохраняет ранее созданные сообщения | `r14-main-shutdown-check` | Исходные ID и список не созданных |
| Полная карта и минимум по оценённой карте | Пока не подтверждено этой тетрадью | Нужны завершённые пилоты, итоговые seed и отдельное подтверждение кандидата |
| Прогноз/управляющее переразмещение | Сейчас не выполняются | TASK §7; никаких метрик обучения |

nbconvert проверяет Python и обработчики, **но не мышь и отображение виджетов в личном браузере Colab**. Тетрадь не утверждает готовность статьи или принятие результатов рецензентом.

In [ ]:
receipt = {'source':SOURCE_SHA,'example_calls':panel.calls,
    'single_replication_not_statistical':panel.last['result']['display_summary']['statistical_status'],
    'saved_status':panel.last['saved']['status'], 'manual_five_checked':True,
    'shutdown_callback_checked':panel.shutdown is not None,
    'full_grid_executed':False,'forecast_executed':False,'colab_browser_checked':False}
print(json.dumps(receipt,ensure_ascii=False,indent=2))